# OSAI Project 1 — Colab v1 Baseline Reproduction

Pascal-VOC 21-class semantic segmentation, ResNet-50 + DeepLabV3+ (OS=16). main 브랜치 기준 v1 baseline 재현.

재현을 위해 다음 항목들을 수행해주셔야 합니다.

1. Drive에 폴더 생성: `MyDrive/osai/p1/input/test_public/` (1000장 jpg)
2. WandB API key Colab Secret에 저장

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 설정 변수

In [2]:
REPO_URL = "https://github.com/geniemo/osai.git"
BRANCH   = "main"
CONFIG   = "src/config/colab.yaml"
DRIVE    = "/content/drive/MyDrive/osai/p1"

import os
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
print(f"Repo: {REPO_URL} (branch={BRANCH})")
print(f"Config: {CONFIG}")
print(f"Drive: {DRIVE}")
assert os.path.exists(f"{DRIVE}/input/test_public"), f"test_public/ not found at {DRIVE}/input/test_public"
n_imgs = len([f for f in os.listdir(f"{DRIVE}/input/test_public") if f.endswith(".jpg")])
print(f"test_public: {n_imgs} jpg files")

Repo: https://github.com/geniemo/osai.git (branch=main)
Config: src/config/colab.yaml
Drive: /content/drive/MyDrive/osai/p1
test_public: 1000 jpg files


## 2. 저장소 clone

In [3]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai/p1

/content
Cloning into 'osai'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'osai/p1'
/content


## 3. uv 설치 + 의존성 sync

In [4]:
!pip install -q uv
!uv sync

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 115.0 MB/s eta 0:00:00
error: No `pyproject.toml` found in current directory or any parent directory


## 4. GPU 확인

In [5]:
!uv run python -c "\
import torch; \
print('Device:', torch.cuda.get_device_name(0)); \
print('Capability:', torch.cuda.get_device_capability(0)); \
print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB'); \
print('PyTorch:', torch.__version__)"

  File "<string>", line 1
    import torch;  print('Device:', torch.cuda.get_device_name(0));  print('Capability:', torch.cuda.get_device_capability(0));  print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB');  print('PyTorch:', torch.__version__)
IndentationError: unexpected indent


## 5. WandB 로그인

In [6]:
from google.colab import userdata
import os; os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

!uv run wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: g1nie (g1nie-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 6. Test images Colab 로컬로 복사

In [7]:
!mkdir -p input
!cp -r {DRIVE}/input/test_public input/
# Colab local로 복사
!ls input/test_public | wc -l
!ls input/test_public | head -3

1000
000.jpg
001.jpg
002.jpg


## 7. 데이터 다운로드 (VOC + COCO)

In [8]:
!uv run python -m src.data.download --voc-root data/voc --coco-root data/coco

/usr/bin/python3: Error while finding module specification for 'src.data.download' (ModuleNotFoundError: No module named 'src')


## 8. COCO mask cache 사전 생성

95K instance mask를 PNG로 1회 생성

In [9]:
!uv run python -m src.build_coco_masks --coco-root data/coco --num-workers 4

/usr/bin/python3: Error while finding module specification for 'src.build_coco_masks' (ModuleNotFoundError: No module named 'src')


## 9. 학습 (Stage 1 + Stage 2 자동, ~10-12h)

ckpt가 Drive에 저장됨. 세션 끊기면 다시 시작 시 자동 resume.

In [10]:
!uv run python -m src.train --config {CONFIG}

/usr/bin/python3: Error while finding module specification for 'src.train' (ModuleNotFoundError: No module named 'src')


## 10. Validation mIoU 측정

최종 ckpt로 val mIoU + 21 class별 IoU 출력.

In [11]:
!uv run python -m src.eval --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

/usr/bin/python3: Error while finding module specification for 'src.eval' (ModuleNotFoundError: No module named 'src')


## 11. TTA Validation mIoU (선택, 실제 inference 성능)

Multi-scale + hflip TTA로 측정.

In [12]:
!uv run python -m src.eval_tta --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

/usr/bin/python3: Error while finding module specification for 'src.eval_tta' (ModuleNotFoundError: No module named 'src')


## 12. 추론 — test_public 1000장 (TTA)

input/test_public/*.jpg → output/pred_FINAL/*.png. ~5–10분 (T4).

In [13]:
!mkdir -p output
!uv run python -m src.infer \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --input input/test_public \
    --output output/pred_FINAL

/usr/bin/python3: Error while finding module specification for 'src.infer' (ModuleNotFoundError: No module named 'src')


## 13. ONNX export (가중치 제거, 채점용 ONNX)

`model_structure.onnx` 생성 (~0.3MB, ≤10MB 채점 한도).

In [14]:
!uv run python -m src.export_onnx \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --out {DRIVE}/model_structure.onnx

/usr/bin/python3: Error while finding module specification for 'src.export_onnx' (ModuleNotFoundError: No module named 'src')


## 14. FLOPs 측정

In [15]:
!uv run python -m src.measure_flops --onnx {DRIVE}/model_structure.onnx

/usr/bin/python3: Error while finding module specification for 'src.measure_flops' (ModuleNotFoundError: No module named 'src')


## 15. PNG zip 생성

In [16]:
!uv run python -m src.package_submission \
    --pred output/pred_FINAL \
    --out {DRIVE}/submission_pred.zip

/usr/bin/python3: Error while finding module specification for 'src.package_submission' (ModuleNotFoundError: No module named 'src')


## 16. 결과물 확인

In [17]:
!ls -la {DRIVE}/
!ls -la {DRIVE}/checkpoints/

total 8
drwx------ 2 root root 4096 Apr 25 05:17 checkpoints
drwx------ 3 root root 4096 Apr 25 05:21 input
total 0
